# LLM Evaluation on Google Colab

Notebook này dùng riêng cho Google Colab để benchmark các LLM trên bài toán comparative quintuple extraction.

Trước khi chạy:
1. Thêm `OPENROUTER_API_KEY` vào Colab Secrets.
2. Chọn một trong hai cách nạp project: clone từ GitHub hoặc upload `project.zip`.
3. Chỉnh `DATASETS`, `SPLIT`, `MODELS`, `PROMPT_STRATEGY` ở cell cấu hình.

In [ ]:
# Cell 1 · Colab paths
import os
import sys

assert 'google.colab' in sys.modules or os.path.exists('/content'), 'Notebook này chỉ dành cho Google Colab.'

WORK_DIR = '/content/msc-project'
LLMEVAL_DIR = os.path.join(WORK_DIR, 'llm_eval')
DATASETS_ROOT = os.path.join(WORK_DIR, 'datasets')
OUTPUT_DIR = os.path.join(LLMEVAL_DIR, 'results')
CACHE_DIR = os.path.join(LLMEVAL_DIR, 'cache')

print('Project root :', WORK_DIR)
print('LLM eval dir :', LLMEVAL_DIR)

In [ ]:
# Cell 2 · Clone repo hoặc upload project.zip
GITHUB_REPO = ''  # ví dụ: 'https://github.com/haiyan/msc-project.git'

import os
import shutil
import subprocess
import zipfile

if not os.path.exists(LLMEVAL_DIR):
    if GITHUB_REPO:
        subprocess.run(['git', 'clone', '--depth', '1', GITHUB_REPO, WORK_DIR], check=True)
        print('Cloned from GitHub.')
    else:
        from google.colab import files
        print('Upload project.zip below:')
        uploaded = files.upload()
        zip_name = list(uploaded.keys())[0]
        with zipfile.ZipFile(zip_name) as zf:
            zf.extractall('/content/')
        extracted = [
            d for d in os.listdir('/content/')
            if os.path.isdir(f'/content/{d}') and d != 'sample_data' and d != 'msc-project'
        ]
        if extracted and not os.path.exists(WORK_DIR):
            shutil.move(f'/content/{extracted[0]}', WORK_DIR)
        print(f'Extracted to {WORK_DIR}')
else:
    print(f'Project already exists at {WORK_DIR}')

In [ ]:
# Cell 3 · Install dependencies
import os
import subprocess
import sys

reqs = os.path.join(LLMEVAL_DIR, 'requirements.txt')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', reqs, '-q'], check=True)
print('Dependencies installed.')

In [ ]:
# Cell 4 · Load OpenRouter API key from Colab Secrets
import os
from google.colab import userdata

os.environ['OPENROUTER_API_KEY'] = userdata.get('OPENROUTER_API_KEY')
assert os.environ.get('OPENROUTER_API_KEY'), 'OPENROUTER_API_KEY not found in Colab Secrets.'
print('API key loaded.')

In [ ]:
# Cell 5 · Configuration
DATASETS = 'camera-coqe,vcom-data'
SPLIT = 'test'
PROMPT_STRATEGY = 'few-shot'  # zero-shot | few-shot | cot

MODELS = [
    'openai/gpt-4o-mini',
    'anthropic/claude-3.5-haiku',
    'google/gemini-2.0-flash-001',
    'deepseek/deepseek-chat',
    'qwen/qwen-2.5-72b-instruct',
    'meta-llama/llama-3.3-70b-instruct',
]

TEMPERATURE = 0.0
MAX_OUTPUT_TOKENS = 256
SLEEP_SECONDS = 0.3
LIMIT = 0

print('Configuration ready.')

In [ ]:
# Cell 6 · Run evaluation
import os
import subprocess
import sys

cmd = [
    sys.executable, os.path.join(LLMEVAL_DIR, 'run_eval.py'),
    '--datasets', DATASETS,
    '--split', SPLIT,
    '--models', *MODELS,
    '--prompt-strategy', PROMPT_STRATEGY,
    '--base-url', 'https://openrouter.ai/api/v1',
    '--api-key-env', 'OPENROUTER_API_KEY',
    '--temperature', str(TEMPERATURE),
    '--max-output-tokens', str(MAX_OUTPUT_TOKENS),
    '--sleep-seconds', str(SLEEP_SECONDS),
    '--datasets-root', DATASETS_ROOT,
    '--output-dir', OUTPUT_DIR,
    '--cache-dir', CACHE_DIR,
]
if LIMIT > 0:
    cmd += ['--limit', str(LIMIT)]

result = subprocess.run(cmd, text=True, capture_output=False)
print('Exit code:', result.returncode)

In [ ]:
# Cell 7 · Show summary
import json
import pathlib

summary_file = pathlib.Path(OUTPUT_DIR) / f'summary__{SPLIT}.json'

if summary_file.exists():
    with open(summary_file, 'r', encoding='utf-8') as f:
        rows = json.load(f)
    try:
        import pandas as pd
        df = pd.DataFrame([
            {
                'dataset': r['dataset'],
                'model': r['model'],
                'E-T5-MACRO-F1': round(r.get('E-T5-MACRO-F1', 0), 4),
                'E-T4-F1': round(r.get('E-T4-F1', 0), 4),
                'E-CEE-MICRO-F1': round(r.get('E-CEE-MICRO-F1', 0), 4),
            }
            for r in rows
        ]).sort_values(['dataset', 'E-T5-MACRO-F1'], ascending=[True, False])
        print(df.to_string(index=False))
    except ImportError:
        print(rows)
else:
    print('Summary file not found.')

In [ ]:
# Cell 8 · Download results zip
import pathlib
import zipfile
from google.colab import files

results_path = pathlib.Path(OUTPUT_DIR)
zip_path = '/content/llm_eval_results.zip'

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in results_path.rglob('*'):
        if f.is_file():
            zf.write(f, f.relative_to(results_path.parent))

files.download(zip_path)
print('Download started.')